In [ ]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:100% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:20pt;}
div.text_cell_render.rendered_html{font-size:18pt;}
div.text_cell_render.rendered_html{font-size:15pt;}
div.output {font-size:18pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:18pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:18pt;padding:5px;}
table.dataframe{font-size:18px;}
</style>
"""))

# <span style="color:red">ch9.07_LangChain과 vectorDatabase을 활용한 RAG 구현_upstage</span>

# RAG 절차
- https://law.go.kr/법령/소득세법 에서 doc다운로드(hwp는 파이썬 못 읽음. pdf는 한글의 경우 짤림) 받아 파일형식을 docx로 변경

1. 문서를 읽는다 (document_loader이용)
2. 읽어온 문서를 쪼갠다(tiktoken 이용)
    - 모델의 context window를 초과 (128,000 context window)
    - 문서가 길면(input이 길면), 비용과 시간이 오래 걸림
3. 쪼갠 문서를 임베딩 -> vector database에 저장 -> chroma(local vector DB), pinecone(클라우드 vector DB)
4. 질문과 vector Database의 유사도 검색
5. 유사도 검색으로 가져온 문서를 LLM에 질문과 같이 전달하여 답변 생성

[ RAG 구현 절차 ]
```
1.	문서의 내용을 읽는다(document_loader를 이용)
(1)	https://python.langchain.com/v0.2/docs/integrations/document_loaders/ 
(2)	https://python.langchain.com/v0.2/docs/integrations/document_loaders/microsoft_word/
%pip install --upgrade --quiet  docx2txt
2.	문서를 쪼갠다(한번에 이해하고 처리할 수 있는 입력+출력 토큰수가 제한)
(1)	 https://python.langchain.com/v0.2/docs/how_to/recursive_text_splitter/#splitting-text-from-languages-without-word-boundaries 
%pip install -qU langchain-text-splitters
3.	쪼갠 문서를 임베딩하여 vector database에 넣음
(1)	OpenAIEmbeddings나 UpstageEmbeddings이용해서 임베딩
(2)	https://python.langchain.com/v0.2/docs/integrations/vectorstores/chroma/  
%pip install –q langchain-chroma
4.	질문을 이용해 유사도 검색
5.	유사도 검색한 문서를 LLM에 질문으로 전달하여 답변 얻음(제공되는 Prompt활용)
(1)	https://python.langchain.com/v0.2/docs/tutorials/rag/
%pip install –q langchain langchainhub

https://smith.langchain.com에서 key생성 .env key 추가
```

# 2. 문서를 쪼개면서 읽기(o)

In [1]:
import time
start = time.time()

from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = Docx2txtLoader('./tax_docs/소득세법(법률)(제20615호)(20250701).docx')
text_splitter = RecursiveCharacterTextSplitter( # 문자 단위로 쪼갬(토큰단위 아님)
    chunk_size=1500, # 문서를 쪼갤때 1500글자씩
    chunk_overlap=200
)
# 1번째 chunk 1~1450글자 (\n이 있는부분까지)
# 2번째 chunk 1250글자~2570글자

documents = loader.load_and_split(text_splitter=text_splitter)

runtime = time.time() - start

print('문서 쪼개면서 읽는 시간 :', runtime)

문서 쪼개면서 읽는 시간 : 5.393361806869507


# 쪼갠 문서를 임베딩 -> 벡터 데이터베이스 저장
- embedding 모델 : upstage solar-embedding-1-large
- 벡터 데이터베이스 : chroma 
- https://python.langchain.com/v0.2/docs/how_to/embed_text/

### 재시작하면 임베딩부터 다시

In [6]:
from dotenv import load_dotenv
from langchain_upstage import UpstageEmbeddings
load_dotenv()

embeddings = UpstageEmbeddings(model="solar-embedding-1-large")

In [4]:
doc_result = embeddings.embed_documents(
    ["소득세법 어쩌구 저쩌구", documents[0].page_content]
)
print(len(doc_result), len(doc_result[0]), len(doc_result[1]))

2 4096 4096


In [16]:
len(embeddings), len(embeddings[0]), len(embeddings[1])

(2, 3072, 3072)

In [18]:
len(embedding.embed_query("소득세"))

3072

### 처음저장할때만 실행

In [7]:
%%time
from langchain_chroma import Chroma
#데이터를 처음 저장할 때
database = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    collection_name="tax_collection", # 생략시 이름 랜덤(불러오기를 위함)
    persist_directory='./chroma_upstage'      # 생략시 로컬데이터베이스에 저장 안됨. 인메모리로만 사용(프로그램 종료시 db 날아감)
)

# # 이미 저장된 vector DB를 사용할 때
# database = Chroma(
#     embedding_function=embeddings,
#     collection_name="tax_collection",
#     persist_directory='./chroma_upstage'
# )

RateLimitError: Error code: 429 - {'error': {'message': "You've reached your API request limit. Please wait and try again later. If your use case requires a higher rate limit, you can request an increase at https://support.upstage.ai. Including your intended use case and expected volume will help us process your request faster", 'type': 'too_many_requests', 'param': '', 'code': 'too_many_requests'}}

# 4. vector DB에 질문과 유사도 검색(답변 생성을 위한 retrieval)

In [22]:
query = "연봉 5000만원인 직장인의 소득세는 얼마인가요?"
retrieved_docs = database.similarity_search(query,
                                           k=3) # 기본 k는 4

In [23]:
retrieved_docs[0]

Document(id='97ae9d3f-adbf-472a-9f08-7975018b240a', metadata={'source': './tax_docs/소득세법(법률)(제20615호)(20250701).docx'}, page_content='바. 「문화유산의 보존 및 활용에 관한 법률」에 따라 국가지정문화유산으로 지정된 서화ㆍ골동품의 양도로 발생하는 소득\n\n사. 서화ㆍ골동품을 박물관 또는 미술관에 양도함으로써 발생하는 소득\n\n아. 제21조제1항제26호에 따른 종교인소득 중 다음의 어느 하나에 해당하는 소득\n\n\u3000\u3000\u3000\u30001) 「통계법」 제22조에 따라 통계청장이 고시하는 한국표준직업분류에 따른 종교관련종사자(이하 “종교관련종사자”라 한다)가 받는 대통령령으로 정하는 학자금\n\n\u3000\u3000\u3000\u30002) 종교관련종사자가 받는 대통령령으로 정하는 식사 또는 식사대\n\n\u3000\u3000\u3000\u30003) 종교관련종사자가 받는 대통령령으로 정하는 실비변상적 성질의 지급액\n\n\u3000\u3000\u3000\u30004) 종교관련종사자 또는 그 배우자의 출산이나 6세 이하(해당 과세기간 개시일을 기준으로 판단한다) 자녀의 보육과 관련하여 종교단체로부터 받는 금액으로서 월 20만원 이내의 금액\n\n\u3000\u3000\u3000\u30005) 종교관련종사자가 기획재정부령으로 정하는 사택을 제공받아 얻는 이익\n\n자. 법령ㆍ조례에 따른 위원회 등의 보수를 받지 아니하는 위원(학술원 및 예술원의 회원을 포함한다) 등이 받는 수당\n\n[전문개정 2009. 12. 31.]\n\n\n\n제13조 삭제 <2009. 12. 31.>\n\n\n\n제2절 과세표준과 세액의 계산 <개정 2009. 12. 31.>\n\n\n\n제1관 세액계산 통칙 <개정 2009. 12. 31.>\n\n\n\n제14조(과세표준의 계산) ① 거주자의 종합소득 및 퇴직소득에 대한 과세표준은 각각 구분하여 계산한다.\n\n②

# 5. 유사도 검색으로 가져온 문서를 질문과 같이 LLM 전달하여 답변 생성

In [24]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-4.1-nano")

In [25]:
prompt = f"""[idendity]
- 당신은 최고의 한국 소득세 전문가입니다.
- [context]를 참고해서 사용자의 질문에 답변해주세요.
[context]는 다음과 같습니다.
{retrieved_docs}
Question : {query}"""

In [26]:
ai_message = llm.invoke(prompt)

In [29]:
print(ai_message.content)

연봉 5,000만원인 직장인의 소득세는 근로소득에 대한 과세 표준을 계산한 후, 이에 맞는 세율을 적용하여 산출합니다. 아래는 일반적인 계산 절차입니다.

1. 근로소득공제 계산
근로소득공제는 연봉에 따라 차등 적용되며, 2023년 기준으로는 다음과 같습니다.

- 연봉 5,000만원인 경우, 근로소득공제액은 약 1,540만원입니다.
(구체적 공제액은 국세청 홈택스 계산기를 참고하세요.)

2. 과세표준 산출
총급여(연봉)에서 근로소득공제액을 차감하여 과세표준을 계산합니다.

- 과세표준 = 50,000,000원 - 15,400,000원 = 34,600,000원

3. 세율 적용
과세표준에 따른 세율은 다음과 같습니다.(2023년 기준)

| 과세표준 구간 (원) | 세율 | 누진공제 (원) |
|-------------------|-------|--------------|
| 1,200만원 이하      | 6%    | 0            |
| 1,200만원 초과 ~ 4,600만원 이하 | 15%   | 108만원      |
| 4,600만원 초과 ~ 8,800만원 이하 | 24%   | 522만원      |
| 8,800만원 초과 ~ 1억5천만원 이하 | 35%   | 1,490만원    |
| 1억5천만원 초과     | 38%   | 2,200만원    |

우리의 과세표준은 34,600,000원으로 1,200만원 초과 ~ 4,600만원 이하 구간에 해당하며, 세율은 15%입니다.

세금 = 과세표준 × 15% - 누진공제액 108만원

= 34,600,000원 × 0.15 - 1,080,000원

= 5,190,000원 - 1,080,000원

= 4,110,000원

4. 주민세
근로소득세의 10%인 주민세(지역세)도 부과됩니다.

주민세 = 4,110,000원 × 10% = 411,000원

5. 총 세액
총 세액 = 근로소득세 + 주민세 = 4,110,000원 + 411,000원 = 4,521,000원

**결론:**  
연봉 5,00

# 5.1 Augmentation을 위한 제공되는 Prompt활용하여 langchain으로 답변 생성

In [30]:
query = "연봉 5000만원인 직장인의 소득세는 얼마인가요?"

from langchain import hub
prompt = hub.pull("rlm/rag-prompt")
prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, metadata={'lc_hub_owner': 'rlm', 'lc_hub_repo': 'rag-prompt', 'lc_hub_commit_hash': '50442af133e61576e74536c6556cefe1fac147cad032f4377b60c436e6cdcb6e'}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:"), additional_kwargs={})])

### RetrievalQA를 통해 LLM전달 (create_retrieval_chain이 대체)
```
  query -> retrievar전달(vector 검색 수행: 최적화된 유사도가 높은 데이터 산출) -> retrieval문서 -> {context}에 삽입
  -> query = prompt의 {question}에 삽입
```

In [32]:
# QA체인만들기
from langchain.chains import RetrievalQA
qa_chain = RetrievalQA.from_chain_type(
    llm,
    retriever = database.as_retriever(search_kwargs={'k':3}),
    chain_type_kwargs={"prompt":prompt}
)

In [33]:
ai_message = qa_chain.invoke({"query":query})

In [36]:
ai_message

{'query': '연봉 5000만원인 직장인의 소득세는 얼마인가요?',
 'result': '연봉 5000만원인 직장인의 소득세는 정확히 계산하기 어렵지만, 일반적으로 근로소득세는 과세표준과 세율에 따라 결정됩니다. 대체로 연봉 5000만원 정도는 24%의 세율 구간에 해당하며, 공제액 등을 고려하면 세금은 약 100만~150만원 수준일 수 있습니다. 정확한 금액은 공제, 소득공제, 세법 변경 등을 반영한 세무 계산이 필요합니다.'}